# Consistency Models

我们上一章中提到的 Reflow 技术与本章提到的 CM 都是加速生成领域的关键技术。关于这两者的地位，工业界大趋势是 Reflow 地位远高于 CM。无论是最终生成质量还是训练难度，如今的前沿模型都很青睐 Reflow。

那为什么还要讲 CM？原因是 CM 的范式非常新颖且异类。我们愿意多看到一些不一样的想法。

推荐你读 https://arxiv.org/abs/2303.01469 Consistency Models 这是一致性模型原论文。值得一提，我们提到的 CM, DDPM 与 Scored-based Models 全部出自 Stanford 的华人宋飏之手。

# 基本逻辑

CM 的设计初衷是加速图像生成，达到原生一步生成图像目的。对于传统的 Diffusion 模型，如 FM 或 DDIM，我们尝试使用 Euler 法或者更高阶的 ODE/SDE 解法逼近方程的解，这注定需要多步生成。CM 提出，模型直接学习一个映射函数，根据时间步 $t$ 和当下图像 $x_t$ 一步输出图像 $x_0$。

回顾一下第四章中的内容。Score-based Models 指出无论 SMLD 还是 DDIM 的前向扩散过程都在模拟 SDE 的解$$\mathrm{d}\mathbf{x} = \mathbf{f}(\mathbf{x}, t)\mathrm{d}t + g(t)\mathrm{d}\mathbf{w}$$
或者写得更加具体 $$\mathrm{d}\mathbf{x}_t = \mu(\mathbf{x_t}, t)\mathrm{d}t + \sigma(t)\mathrm{d}\mathbf{w_t}$$
其中 $t \in [0, T]$，$T > 0$ 是一个常数， $\mu(\cdot, \cdot)$ 与 $\sigma(\cdot)$ 分别是漂移项系数与扩散项系数。$\{\mathbf{w}_t\}_{t \in [0,T]}$ 则是 Wiener 过程。

我们记 $\mathbf{x}_t$ 为 $p_t(\mathbf{x})$，那么$p_0(\mathbf{x}) \equiv p_{\text{data}}(\mathbf{x})$。一个关于以上 SDE 的性质是，存在一个对应的 ODE，使得其的解恰好符合分布 $p_t(\mathbf{x})$。这个 ODE 我们非常熟悉，就是在第四章中提到的逆向过程对应的 ODE$$d\mathbf{x}_t = \left[ \boldsymbol{\mu}(\mathbf{x}_t, t) - \frac{1}{2} \sigma(t)^2 \nabla \log p_t(\mathbf{x}_t) \right] dt.$$

以上这个 ODE 非常重要。我们称其为 PF ODE (Probability Flow ODE)。CM 的核心思想是观察这个 ODE 的轨迹。

我们希望模型学习一个函数，名为一致性函数。这个函数获得 PF ODE 轨迹上任意一点信息都可以直接给出原始图像，达到一步生成图像的目的。

我们正式给出一致性函数的定义。对于 PF ODE 轨迹上的任意两点 $(\mathbf{x}_t, t)$ 和 $(\mathbf{x}_{t'}, t')$，一致性函数 $f$ 必须满足 $$f(\mathbf{x}_t, t) = f(\mathbf{x}_{t'}, t') = \mathbf{x}_\epsilon$$
这里的 $\epsilon$ 是一个设定的微小数值，因此 $\mathbf{x}_\epsilon$ 是极其靠近 $x_0$ 的一点。

更多的，一致性函数必须满足一个边界条件。当 $t = \epsilon$ 时，$f(\mathbf{x}_\epsilon, \epsilon) = \mathbf{x}_\epsilon$。这是为了保证在接近原图时进行恒等映射。更多的，神经网络预测在 $t \to 0$ 时非常不稳定甚至趋于无穷大，在时间步接近初始时我们人为留下一些缓冲区域。后续我们会详谈这个边界条件的处理。

下面这张图展示了 CM 所做的事。实际上是观察 PF ODE 轨迹直接给出 $x_0$。

<img src="./assets/CM.png" width="600" height="280">

在实际工程实现中我们发现，模型直接学习一致性函数本身是困难的，尤其是为了满足边界条件时的恒等映射输出。因此，我们给出两种办法变换神经网络函数到原一致性函数。

记神经网络拟合函数是 $F_\theta(\cdot,\cdot)$。第一种函数方案是 $$f_\theta(\mathbf{x}, t) = \begin{cases} \mathbf{x} & t = \epsilon \\ F_\theta(\mathbf{x}, t) & t \in (\epsilon, T] \end{cases}$$这种方案缺点是在 $t = \epsilon$ 处不可微，会导致训练时在这个临界点出现震荡。

第二种方案则是更主流的方案 $$f_\theta(\mathbf{x}, t) = c_{skip}(t)\mathbf{x} + c_{out}(t)F_\theta(\mathbf{x}, t)$$ 
通过设计 $c_{skip}(t)$ 和 $c_{out}(t)$，使得当 $t = \epsilon$ 时，$c_{skip}(\epsilon) = 1$ 且 $c_{out}(\epsilon) = 0$。这种方案优点是整个网络在全时域是连续且可微的。这允许我们直接套用现有的 U-Net 或 Transformer 架构作为基础，而无需修改内部结构。

关于具体的函数设计，主流思路是设定 $\sigma_{data}$ 为数据的标准差 (通常取 0.5)，则系数设计如下 $$c_{skip}(t) = \frac{\sigma_{data}^2}{(t - \epsilon)^2 + \sigma_{data}^2}$$ $$c_{out}(t) = \frac{(t - \epsilon) \cdot \sigma_{data}}{\sqrt{(t - \epsilon)^2 + \sigma_{data}^2}}$$
在 $t$ 增大时，会让 $c_{skip}$ 呈平方级衰减，原因是在高噪声下，$c_{out}$ 占据主导地位。

为了配合 $c_{skip}$ 和 $c_{out}$，神经网络 $F_\theta$ 的输入 $\mathbf{x}$ 通常也会经过一个缩放系数 $c_{in}(t)$ $$F_\theta(c_{in}(t) \cdot \mathbf{x}, t)$$ $$c_{in}(t) = \frac{1}{\sqrt{t^2 + \sigma_{data}^2}}$$

关于常数 $T$ 与 $\epsilon$ 选择，原文中给出的方案是 $T = 80$，$\epsilon = 0.002$。这个数值非常合理，我们必须保证 $\epsilon$ 不至于被浮点数误差忽略也不会影响最终生成质量。

# 推理

我们先来说说推理，原因是训练比较技巧性。虽然 CM 模型的设计初衷是一步生成图像，但是原文作者给出一种多步修正的推理算法。

设定初始噪声服从高斯分布 $\hat{\mathbf{x}}_T \sim \mathcal{N}(\mathbf{0}, T^2 \mathbf{I})$，已得到的一致性函数 $f_{\theta}(\cdot, \cdot)$，时间点序列 $\tau_1 > \tau_2 > \dots > \tau_{N-1}$。

初始步 $\mathbf{x} \leftarrow \boldsymbol{f}_{\theta}(\hat{\mathbf{x}}_T, T)$。

重复以下步骤 $N-1$ 次。采样噪声 $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$。前向加噪 $\hat{\mathbf{x}}_{\tau_n} \leftarrow \mathbf{x} + \sqrt{\tau_n^2 - \epsilon^2} \mathbf{z}$。再次推理初始图像 $\mathbf{x} \leftarrow \boldsymbol{f}_{\theta}(\hat{\mathbf{x}}_{\tau_n}, \tau_n)$。

最终输出最后一步给出的图像 $\mathbf{x}$。完整算法如下。

$$\begin{aligned}
&\textbf{Algorithm 1} \text{ Multistep Consistency Sampling} \\
\hline
&\textbf{Input: } \text{Consistency model } \boldsymbol{f}_{\theta}(\cdot, \cdot), \text{ sequence of time points } \tau_1 > \tau_2 > \dots > \tau_{N-1}, \\
&\quad \text{initial noise } \hat{\mathbf{x}}_T \\
&\mathbf{x} \leftarrow \boldsymbol{f}_{\theta}(\hat{\mathbf{x}}_T, T) \\
&\textbf{for } n = 1 \textbf{ to } N - 1 \textbf{ do} \\
&\quad \text{Sample } \mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}) \\
&\quad \hat{\mathbf{x}}_{\tau_n} \leftarrow \mathbf{x} + \sqrt{\tau_n^2 - \epsilon^2} \mathbf{z} \\
&\quad \mathbf{x} \leftarrow \boldsymbol{f}_{\theta}(\hat{\mathbf{x}}_{\tau_n}, \tau_n) \\
&\textbf{end for} \\
&\textbf{Output: } \mathbf{x} \\
\hline
\end{aligned}$$

为什么要反复加噪再去噪？其实根据 CM 原生一步出图的设计，我们仅仅去噪一次就可以得到最终图像了。但是多次重复则可以达到更佳的效果。这本质类似 Langevin Dyanmics，我们尝试通过多次采样达到无偏结果。

CM 在推理上类似 Diffusion。同样的，其在针对图像生成任务进行训练之后，不需要任何后训练即可胜任图像补完等等任务。

# 训练

CM 的训练分两种方式。首先是蒸馏现有的 Diffusion 模型，其次是孤立的学习。第一种方式是绝对的主流。我们详谈。

### Consistency Distillation

CM 的蒸馏式训练非常有趣。简而言之，我们先让 CM 根据某一时间步下图像预测原始图像，再利用教师 Diffusion 模型预测同一时间步下的下一个时间步的图像，再让 CM 根据新的时间步下的图像预测原始图像。我们希望这两次预测结果尽量一致。我们展开说说。

我们记教师模型的预测结果为 $\Phi(\cdot,\cdot)$，接收一个图像 $x$ 和时间步 $t$ 给出该点分数 (梯度) 或者矢量场。

对于时间段 $[\epsilon, T]$，我们选取 $N$ 个时间点 $t_1 = \epsilon < t_2 < ... < t_N = T$。

关于时间步具体选取方法，如下 $$t_i = \left( \epsilon^{1/\rho} + \frac{i-1}{N-1} (T^{1/\rho} - \epsilon^{1/\rho}) \right)^\rho$$

其中默认 $\rho = 7$，$T = 80$，$\epsilon = 0.002$。

对于时间点 $t_{n+1}$ 由原图向前加噪得到的图像 $\mathbf{x}_{t_{n+1}}$，我们计算 $$\hat{\mathbf{x}}_{t_n}^\phi := \mathbf{x}_{t_{n+1}} + (t_n - t_{n+1})\Phi(\mathbf{x}_{t_{n+1}}, t_{n+1}; \phi)$$
这就是利用教师模型进行一次 Euler 法推理，可以将 $\hat{\mathbf{x}}_{t_n}^\phi$ 理解为沿着 PF ODE 轨迹预测的下一个点。

现在我们定义损失函数 $$\mathcal{L}_{CD}^N(\boldsymbol{\theta}, \boldsymbol{\theta}^-; \phi):= 
\mathbb{E}_{\substack{\mathbf{x} \sim p_{\text{data}}, n \sim \mathcal{U}[\![1, N-1]\!], \mathbf{x}_{t_{n+1}} \sim \mathcal{N}(\mathbf{x}; t_{n+1}^2 \mathbf{I})}} \left[ \lambda(t_n) d\left(\boldsymbol{f}_{\boldsymbol{\theta}}(\mathbf{x}_{t_{n+1}}, t_{n+1}), \boldsymbol{f}_{\boldsymbol{\theta}^-}(\hat{\mathbf{x}}_{t_n}^{\phi}, t_n)\right) \right]$$
这个损失函数非常直观。我们希望模型在同一条 PF ODE 路径上预测的结果都相同。

解释以下为什么出现了 $\theta$ 与 $\theta ^ -$。作者为了训练稳定性分别使用了 EMA 权重 $\theta ^ -$ 与实时权重 $\theta$。如果全部采用实时权重计算，模型非常容易作弊，因为只需要把所有结果全部输出为 $0$，距离计算也会降低到最小。

更具体的，EMA 计算方式是对于 $ 0 \leq \mu < 1$，有 $$\boldsymbol{\theta}^- \leftarrow \operatorname{stopgrad}(\mu \boldsymbol{\theta}^- + (1 - \mu)\boldsymbol{\theta}).$$

我们继续解释这个损失函数。$d(\cdot,\cdot)$ 是一个度量。这里的度量可以是 $L_1$ 度量或 $L_2$ 度量或 Learned Perceptual Image Patch Similarity 度量。最终作者发现 $LPIPS$ 度量效果最好。

$\lambda(t_n)$ 则是一个权重函数，为不同时间步分配权重，推荐 $\lambda(t_n) \equiv 1$。

所以完整的训练流程是，假如有一个 Batch 的数据集。我们挑出一张图片 $x$，随机选择一个正整数 $ 1 \leq n \leq N$。前向加噪 $$\mathbf{x}_{t_{n+1}} = \mathbf{x} + t_{n+1}\mathbf{z}, \quad \mathbf{z} \sim \mathcal{N}(0, \mathbf{I})$$
CM 使用 EMA 权重预测，再由教师模型如上给出 $\hat{\mathbf{x}}_{t_n}^\phi$，CM 使用实时权重对新的图像再次预测。最终计算 $$\mathcal{L}(\theta, \theta^-; \phi) = \lambda(t_n) d\left( f_\theta(\mathbf{x}_{t_{n+1}}, t_{n+1}), f_{\theta^-}(\hat{\mathbf{x}}_{t_n}^\phi, t_n) \right)$$
累积一个 Batch 的损失计算加权平均，反向传播更新参数。

完整训练算法如下。

$$\begin{array}{l}
\hline
\textbf{Algorithm 2 } \text{Consistency Distillation (CD)} \\
\hline
\textbf{Input: } \text{dataset } \mathcal{D}, \text{initial model parameter } \boldsymbol{\theta}, \text{learning rate} \\
\eta, \text{ ODE solver } \Phi(\cdot, \cdot; \phi), d(\cdot, \cdot), \lambda(\cdot), \text{ and } \mu \\
\boldsymbol{\theta}^- \leftarrow \boldsymbol{\theta} \\
\textbf{repeat} \\
\quad \text{Sample } \mathbf{x} \sim \mathcal{D} \text{ and } n \sim \mathcal{U}[\![1, N-1]\!] \\
\quad \text{Sample } \mathbf{x}_{t_{n+1}} \sim \mathcal{N}(\mathbf{x}; t_{n+1}^2 \mathbf{I}) \\
\quad \hat{\mathbf{x}}_{t_n}^{\phi} \leftarrow \mathbf{x}_{t_{n+1}} + (t_n - t_{n+1})\Phi(\mathbf{x}_{t_{n+1}}, t_{n+1}; \phi) \\
\quad \mathcal{L}(\boldsymbol{\theta}, \boldsymbol{\theta}^-; \phi) \leftarrow \\
\quad \quad \lambda(t_n)d(\boldsymbol{f}_{\boldsymbol{\theta}}(\mathbf{x}_{t_{n+1}}, t_{n+1}), \boldsymbol{f}_{\boldsymbol{\theta}^-}(\hat{\mathbf{x}}_{t_n}^{\phi}, t_n)) \\
\quad \boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \eta \nabla_{\theta} \mathcal{L}(\boldsymbol{\theta}, \boldsymbol{\theta}^-; \phi) \\
\quad \boldsymbol{\theta}^- \leftarrow \operatorname{stopgrad}(\mu \boldsymbol{\theta}^- + (1 - \mu)\boldsymbol{\theta}) \\
\textbf{until } \text{convergence} \\
\hline
\end{array}$$

但是，为什么这样训练是合理的？接下来我为你证明这件事。

令 $\Delta t := \max_{n \in [\![1, N-1]\!]} \{ |t_{n+1} - t_n| \}$，且 $f(\cdot, \cdot; \phi)$ 为教师模型 PF ODE 的一致性函数。假设 $f_{\boldsymbol{\theta}}$ 满足 Lipschitz 条件，且对于所有的 $n \in [\![1, N-1]\!]$ 与任意 $x$，在 $t_{n+1}$ 处调用的 ODE 求解器产生的局部误差 $\| \hat{\mathbf{x}}_{t_n}^\phi - \mathbf{x}_{t_n} \|$ 一致有界于 $O((t_{n+1} - t_n)^{p+1})$，其中 $p \geqslant 1$。那么，如果 $\mathcal{L}_{CD}^N(\boldsymbol{\theta}, \boldsymbol{\theta}^-; \phi) = 0$，我们有 $$\sup_{n, \mathbf{x}} \| \boldsymbol{f}_{\boldsymbol{\theta}}(\mathbf{x}, t_n) - \boldsymbol{f}(\mathbf{x}, t_n; \phi) \|_2 = O((\Delta t)^p).$$
换句话说，在训练达到理想最优情况下，CM 学习的函数会拟合到真实 PF ODE 轨迹上的一致性函数，从而达到学习的目的。

为你证明。定义在时间步 $t_n$ 时的全局误差 $E_n$ 为 $$E_n = \sup_{\mathbf{x}} \|f_\theta(\mathbf{x}, t_n) - f(\mathbf{x}, t_n; \phi)\|_2$$
我们的目标是证明 $E_N = O(\Delta t^p)$。

我们对 $n$ 归纳证明这一点。在时间步 $t_0 = \epsilon$ 时，根据边界条件 $f_\theta(\mathbf{x}, \epsilon) = \mathbf{x}$ 以及  $f(\mathbf{x}, \epsilon; \phi) = \mathbf{x}$。因此，$E_0 = 0$。此时成立。

假设在第 $n$ 步时，误差为 $E_n$。我们要考察第 $n+1$ 步的误差 $E_{n+1}$。由 $$f_\theta(\mathbf{x}_{t_{n+1}}, t_{n+1}) = f_\theta(\hat{\mathbf{x}}_{t_n}^\phi, t_n)$$
我们知道$$E_{n+1} = \|f_\theta(\mathbf{x}_{t_{n+1}}, t_{n+1}) - f(\mathbf{x}_{t_{n+1}}, t_{n+1}; \phi)\|$$
代入得到$$E_{n+1} = \|f_\theta(\hat{\mathbf{x}}_{t_n}^\phi, t_n) - f(\mathbf{x}_{t_{n+1}}, t_{n+1}; \phi)\|$$

根据 PF ODE 一致性函数的定义，真实的终点映射 $f$ 在同一条轨迹上是常数。因此$$f(\mathbf{x}_{t_{n+1}}, t_{n+1}; \phi) = f(\mathbf{x}_{t_n}, t_n; \phi)$$代入上式得到 $$E_{n+1} = \|f_\theta(\hat{\mathbf{x}}_{t_n}^\phi, t_n) - f(\mathbf{x}_{t_n}, t_n; \phi)\|$$

根据三角不等式与 Lipschitz 条件 $$E_{n+1} \le {\|f_\theta(\hat{\mathbf{x}}_{t_n}^\phi, t_n) - f_\theta(\mathbf{x}_{t_n}, t_n)\|} + {\|f_\theta(\mathbf{x}_{t_n}, t_n) - f(\mathbf{x}_{t_n}, t_n; \phi)\|} \le  L \cdot \|\hat{\mathbf{x}}_{t_n}^\phi - \mathbf{x}_{t_n}\| + E_n$$
其中 $\|\hat{\mathbf{x}}_{t_n}^\phi - \mathbf{x}_{t_n}\|$ 正是 ODE 的求解器局部误差。因此 $$E_{n+1} \le E_n + L \cdot C \cdot (\Delta t)^{p+1}$$
累加得到 $$E_N \le \sum_{n=0}^{N-1} L \cdot C \cdot (\Delta t)^{p+1} = N \cdot L \cdot C \cdot (\Delta t)^{p+1} = T \cdot L \cdot C \cdot (\Delta t)^{p}$$
得证 $$E_N = O(\Delta t^p)$$

最后我想提醒一件事，如果选用蒸馏教师模型训练 CM 的方案，教师模型务必选取 Flow Matching 模型。虽然原论文认为 CM 可以学习 DDIM 或 EDM 在 ODE 形式下的的路径，但是学习一个弯曲的路径终点映射绝对困难于学习直线路径的终点映射。

### Consistency Training

孤立训练 CM 相对蒸馏教师模型训练会更困难，原因是蒸馏教师模型原生给出了 PF ODE 的轨迹，然而在孤立训练中却需要 CM 自己去寻找。更多的，实际环境中蒸馏训练 CM 的性价比远高于孤立训练。因此我们在此着墨略少。

相对 CD，CT 的核心改动就是，我们直接取出两个相邻时间步的图像给模型做预测，然后计算两个预测之间的距离损失。

我们完整叙述一遍。CT 为了保证训练效率，加入了一个步数调度器 $N(k) = \lceil \sqrt{k \cdot C} \rceil$ 随着训练步数增加而增加时间步的切分精细程度。在训练步数为 $k$ 情况下，对于时间段 $[\epsilon, T]$，我们选取 $N(k)$ 个时间点 $t_1 = \epsilon < t_2 < ... < t_{N(k)} = T$。

对于一个 Batch 的数据集，取出一张原图 $\mathbf{x}$，采样一个噪声 $\mathbf{z} \sim \mathcal{N}(0, \mathbf{I})$，采样一个时间步 $n \sim \mathcal{U}[\![1, N(k)-1]\!]$。 

利用同一个噪声前向加噪出两个相邻时间步图像数据 $\mathbf{x}_{t_{n+1}} = \mathbf{x} + t_{n+1}\mathbf{z}$ 与 $\mathbf{x}_{t_n} = \mathbf{x} + t_n\mathbf{z}$。

分别用模型的实时权重与 EMA 权重预测得到 $f_{\theta^-}(\mathbf{x} + t_n \mathbf{z}, t_n)$ 与 $f_\theta(\mathbf{x} + t_{n+1} \mathbf{z}, t_{n+1})$。

计算损失函数 $$\mathcal{L}(\boldsymbol{\theta}, \boldsymbol{\theta}^-) = \lambda(t_n)d(\boldsymbol{f}_{\boldsymbol{\theta}}(\mathbf{x} + t_{n+1}\mathbf{z}, t_{n+1}), \boldsymbol{f}_{\boldsymbol{\theta}^-}(\mathbf{x} + t_n\mathbf{z}, t_n))$$

遍历一个 Batch 之后计算加权损失，反向传播更新参数。

完整训练算法如下。

$$\begin{array}{l}
\hline
\textbf{Algorithm 3 } \text{Consistency Training (CT)} \\
\hline
\textbf{Input: } \text{dataset } \mathcal{D}, \text{initial model parameter } \boldsymbol{\theta}, \text{learning rate} \\
\eta, \text{ step schedule } N(\cdot), \text{ EMA decay rate schedule } \mu(\cdot), \\
d(\cdot, \cdot), \text{ and } \lambda(\cdot) \\
\boldsymbol{\theta}^- \leftarrow \boldsymbol{\theta} \text{ and } k \leftarrow 0 \\
\textbf{repeat} \\
\quad \text{Sample } \mathbf{x} \sim \mathcal{D}, \text{ and } n \sim \mathcal{U}[\![1, N(k)-1]\!] \\
\quad \text{Sample } \mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}) \\
\quad \mathcal{L}(\boldsymbol{\theta}, \boldsymbol{\theta}^-) \leftarrow \\
\quad \quad \lambda(t_n)d(\boldsymbol{f}_{\boldsymbol{\theta}}(\mathbf{x} + t_{n+1}\mathbf{z}, t_{n+1}), \boldsymbol{f}_{\boldsymbol{\theta}^-}(\mathbf{x} + t_n\mathbf{z}, t_n)) \\
\quad \boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \eta \nabla_{\theta} \mathcal{L}(\boldsymbol{\theta}, \boldsymbol{\theta}^-) \\
\quad \boldsymbol{\theta}^- \leftarrow \operatorname{stopgrad}(\mu(k)\boldsymbol{\theta}^- + (1 - \mu(k))\boldsymbol{\theta}) \\
\quad k \leftarrow k + 1 \\
\textbf{until } \text{convergence} \\
\hline
\end{array}$$

# 总结

本章我们介绍了 Consistency Models，这是加速生成领域的重要成果。CM 在保持一定图像生成质量的同时，将图像生成步数直接压缩到了 $1 \sim 5$ 步，这种效果非常强大。

不过 CM 的缺陷也很明显。由于模型学习了一个极其直接的映射且推理步数极少，导致生成图像往往多样性不足。更多的，训练时需要挂载一个额外的教师模型以及一系列评判模型 (如 $LPIPS$)，显存压力并不小。

一个符合直觉且反直觉的事实是，CM 相比我们之前提到的 DDPM, DDIM, SMLD 与 FM 是个异类。原因是 CM 直接预测 PF ODE 的积分结果，我们之前提到的众多 Diffusion 模型却是预测微分结果。CM 给出一个终点，然而 FM 给出矢量场，SMLD 给出梯度方向，DDPM 与 DDIM 给出噪声。

我们这里提醒关于 DDIM 与 DDPM 的一点，他们预测噪声看似等价直接预测结果，但是我们更喜欢说他们预测了一个瞬时噪声。从步长的角度来看，DDIM 预测的噪声乘上步长系数，才得到其推理公式中的方向项。更多的，我们之前证明过噪声与梯度之间等价关系，预测瞬时噪声本质只是指出演变方向而不是直接预测结果。

所以现在我们知道了加速生成领域的 Reflow 与 CM，但是这远远不是一个领域的全部。现在所说的方法全部属于蒸馏与后训练。然而，加速生成还有其他重大分支，具体大约是 Cache, Sparse Attention, Quantization, 算法的优化以及架构的转移和诸多 Training-free 方法。

我们简单说说。Cache 就是计算缓存方法，通过缓存张量减少模型推理计算量；Sparse Attention 指的是关于注意力架构的改动，如线性注意力等等；Quantization 则是通过降低推理时张量精度减少计算量与显存压力方法。以上三种属于推理或者训练技术，他们的目的是减少资源消耗。

什么是 Training-free？简而言之就是免训练加速生成，如 Layer Skipping 技术，跳过简单任务时模型中的一些层从而加速；或者纯粹改进 ODE 推理方式，数学上加速。

关于算法的优化，我们指对于模型学习目标函数与训练方式的优化，这也是根本性的。我们会在后续为你介绍 Meanflow 等等优化。

最后的，架构的转移则是指直接更换模型架构从而原生加速生成，重要成果有 Visual Autoregressive Models，也就是遵从大模型自回归式地生成图像。这个成果在 2024 年拿到了 NeurIPS 的 Best Paper Award。我们在下一章详谈。